# 10 — Singapore Vehicle Labelling (Grounding DINO)

Auto-labels raw LTA camera images with Singapore-specific vehicle classes using Grounding DINO.

**10-class taxonomy:**
```
0  car             sedan, hatchback, SUV, MPV
1  motorcycle      standard motorcycle
2  scooter         moped, small scooter
3  bus             double-decker and single-decker
4  van             panel van, minivan, delivery van
5  lorry           light lorry, pickup truck
6  container_truck articulated lorry with shipping container
7  prime_mover     tractor unit only
8  tipper_truck    tipper, dump truck, construction truck
9  taxi            Singapore taxi
```

**Drive structure:**
```
sg_smart_city/
├── data/
│   ├── raw/YYYY-MM-DD/<camera_id>/HH-MM-SS.jpg   ← input
│   └── silver/gdino_v1/                           ← output (this notebook)
│       ├── labels/   ← YOLO .txt (auto-accepted ≥0.45)
│       ├── review/   ← annotated JPEGs for human review
│       ├── manifest.csv
│       └── classes.txt
```

**Pipeline:**
1. Mount Drive → load raw images from `data/raw/`
2. Sample 20 images per camera (~1,800 total across 90 cameras)
3. Run Grounding DINO (GPU) with Singapore text prompts
4. Auto-accept ≥0.45 confidence → YOLO `.txt` labels → `data/silver/gdino_v1/labels/`
5. Write 0.25–0.45 confidence → review queue → `data/silver/gdino_v1/review/`

**Runtime:** ~60–90 min for a full 90-camera sample on T4 GPU.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers accelerate Pillow torch torchvision

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT     = Path('/content/drive/MyDrive/sg_smart_city')
RAW_IMAGES_DIR = DRIVE_ROOT / 'data' / 'raw_adversarial'           # night/sunrise collection run
OUTPUT_DIR     = DRIVE_ROOT / 'data' / 'silver' / 'gdino_v1_adversarial'  # separate from original labels

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Raw images:', RAW_IMAGES_DIR)
print('Output:    ', OUTPUT_DIR)
print('Exists:    ', RAW_IMAGES_DIR.exists())

In [ ]:
# ── Cell 3: Sample strategy ───────────────────────────────────────────────────
# For adversarial (night/sunrise) collection: sample 40 per camera instead of 20
# to get better coverage across the full 3-hour window (5am → 8am SGT).

import random
from collections import defaultdict

random.seed(42)

IMAGES_PER_CAMERA = 40  # up from 20 — 3h window needs denser sampling

all_imgs = sorted(RAW_IMAGES_DIR.rglob('*.jpg'))
print(f'Total images found: {len(all_imgs):,}')

# Group by camera_id — p.parent.name is the camera directory (e.g. '1701')
by_camera = defaultdict(list)
for p in all_imgs:
    cam_id = p.parent.name
    by_camera[cam_id].append(p)

sampled = []
for cam_id, paths in by_camera.items():
    sampled.extend(random.sample(paths, min(IMAGES_PER_CAMERA, len(paths))))

random.shuffle(sampled)
print(f'Cameras found : {len(by_camera)}')
print(f'Sampled images: {len(sampled)}')

In [ ]:
# ── Cell 4: Load Grounding DINO ───────────────────────────────────────────────
import torch
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

MODEL_ID = 'IDEA-Research/grounding-dino-base'
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()
print('Grounding DINO loaded.')

In [ ]:
# ── Cell 5: Define classes and inference helpers ───────────────────────────────
import json, csv
from PIL import Image, ImageDraw, ImageFont

SG_CLASSES = [
    'car', 'motorcycle', 'scooter', 'bus', 'van',
    'lorry', 'container_truck', 'prime_mover', 'tipper_truck', 'taxi',
]

GDINO_QUERY = (
    'car . motorcycle . scooter . moped . bus . double decker bus . '
    'van . delivery van . lorry . light truck . pickup truck . '
    'container truck . articulated lorry . prime mover . tractor unit . '
    'tipper truck . dump truck . construction truck . taxi . cab .'
)

TOKEN_TO_CLASS = {
    'car': 0,
    'motorcycle': 1, 'motorbike': 1,
    'scooter': 2, 'moped': 2,
    'bus': 3, 'double decker bus': 3,
    'van': 4, 'delivery van': 4,
    'lorry': 5, 'light truck': 5, 'pickup truck': 5,
    'container truck': 6, 'articulated lorry': 6,
    'prime mover': 7, 'tractor unit': 7,
    'tipper truck': 8, 'dump truck': 8, 'construction truck': 8,
    'taxi': 9, 'cab': 9,
}

COLORS = ['#4fc3f7','#ff8a65','#ce93d8','#a5d6a7','#fff176',
          '#ffab91','#ef9a9a','#80cbc4','#bcaaa4','#f48fb1']

CONF_AUTO   = 0.30   # lowered from 0.45 — mean class conf is 0.31–0.40, need more training samples
CONF_REVIEW = 0.25


def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


def infer(image: Image.Image) -> list[dict]:
    inputs = processor(images=image, text=GDINO_QUERY, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    results = processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=0.20, text_threshold=0.20,
        target_sizes=[image.size[::-1]],
    )[0]
    dets = []
    for box, score, label in zip(results['boxes'].cpu().tolist(),
                                  results['scores'].cpu().tolist(),
                                  results['labels'], strict=False):
        text = label.strip().lower()
        cls_idx = TOKEN_TO_CLASS.get(text)
        if cls_idx is None:
            for tok, idx in TOKEN_TO_CLASS.items():
                if tok in text or text in tok:
                    cls_idx = idx
                    break
        if cls_idx is None or float(score) < CONF_REVIEW:
            continue
        dets.append({'cls': cls_idx, 'label': SG_CLASSES[cls_idx],
                     'conf': round(float(score), 4), 'box_xyxy': box})
    return dets


def to_yolo(det, w, h):
    x1, y1, x2, y2 = det['box_xyxy']
    return f"{det['cls']} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}"


def draw_boxes(image, dets):
    img = image.copy().convert('RGB')
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 13)
    except OSError:
        font = ImageFont.load_default()
    for d in dets:
        c = hex_to_rgb(COLORS[d['cls']])
        w = 2 if d['conf'] >= CONF_AUTO else 1
        draw.rectangle(d['box_xyxy'], outline=c, width=w)
        draw.text((d['box_xyxy'][0]+2, d['box_xyxy'][1]+2),
                  f"{d['label']} {d['conf']:.2f}", fill=c, font=font)
    return img


print('Helpers defined. CONF_AUTO =', CONF_AUTO, ' CONF_REVIEW =', CONF_REVIEW)

In [ ]:
# ── Cell 6: Run labelling ─────────────────────────────────────────────────────
# Outputs:
#   OUTPUT_DIR/labels/<stem>.txt   YOLO format (auto-accepted only)
#   OUTPUT_DIR/review/<name>.jpg   Annotated image (ALL detections)
#   OUTPUT_DIR/manifest.csv        Every detection with confidence
#   OUTPUT_DIR/classes.txt         Class name list
#
# Label stem format: cam{camera_id}_{YYYYMMDD}_{HHMMSS}
# e.g. raw/2026-03-09/1001/14-12-54.jpg → cam1001_20260309_141254.txt
# This avoids filename collisions when multiple cameras are swept at the same time.

labels_dir = OUTPUT_DIR / 'labels'
review_dir = OUTPUT_DIR / 'review'
labels_dir.mkdir(exist_ok=True)
review_dir.mkdir(exist_ok=True)

def img_stem(p: Path) -> str:
    """Unique stem: cam{id}_{YYYYMMDD}_{HHMMSS} — matches yolo_dataset label convention."""
    cam_id  = p.parent.name                          # e.g. '1001'
    date    = p.parent.parent.name.replace('-', '')  # '2026-03-09' → '20260309'
    time_s  = p.stem.replace('-', '')                # '14-12-54'   → '141254'
    return f'cam{cam_id}_{date}_{time_s}'

manifest = []
auto_n = review_n = skip_n = 0

for i, img_path in enumerate(sampled):
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f'  skip {img_path.name}: {e}')
        continue

    dets = infer(image)

    if not dets:
        skip_n += 1
        continue

    stem        = img_stem(img_path)
    auto_dets   = [d for d in dets if d['conf'] >= CONF_AUTO]
    review_dets = [d for d in dets if d['conf'] <  CONF_AUTO]

    # Write YOLO labels for auto-accepted detections
    if auto_dets:
        with open(labels_dir / f'{stem}.txt', 'w') as f:
            for d in auto_dets:
                f.write(to_yolo(d, image.width, image.height) + '\n')
        auto_n += 1

    # Write annotated review image for everything
    draw_boxes(image, dets).save(review_dir / f'{stem}.jpg')
    if review_dets:
        review_n += 1

    # Manifest
    for d in dets:
        manifest.append({
            'image':        f'{stem}.jpg',
            'camera_id':    img_path.parent.name,
            'date':         img_path.parent.parent.name,
            'source_path':  str(img_path),
            'class':        d['label'],
            'cls_idx':      d['cls'],
            'conf':         d['conf'],
            'auto_accepted': d['conf'] >= CONF_AUTO,
            'box':          json.dumps([round(v, 1) for v in d['box_xyxy']]),
        })

    if (i + 1) % 100 == 0:
        print(f'[{i+1}/{len(sampled)}]  auto={auto_n}  review={review_n}  skip={skip_n}')

# Write manifest and class list
with open(OUTPUT_DIR / 'manifest.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['image','camera_id','date','source_path','class','cls_idx','conf','auto_accepted','box'])
    w.writeheader(); w.writerows(manifest)

(OUTPUT_DIR / 'classes.txt').write_text('\n'.join(SG_CLASSES) + '\n')

print('─' * 50)
print(f'Total images  : {len(sampled)}')
print(f'Auto-labelled : {auto_n}')
print(f'Review queue  : {review_n}  →  {review_dir}')
print(f'No detections : {skip_n}')
print(f'Total dets    : {len(manifest)}')

In [ ]:
# ── Cell 7: Review summary — what needs human attention ───────────────────────
import pandas as pd

df = pd.read_csv(OUTPUT_DIR / 'manifest.csv')

print('=== Detection counts by class ===')
print(df.groupby('class')['conf'].agg(['count', 'mean']).round(3).sort_values('count', ascending=False))

print('\n=== Auto-accepted vs review queue ===')
print(df['auto_accepted'].value_counts())

print('\n=== Images most in need of review (lowest avg confidence) ===')
review_needed = (
    df[~df['auto_accepted']]
    .groupby('image')['conf'].mean()
    .sort_values()
    .head(20)
)
print(review_needed.to_string())

In [ ]:
# ── Cell 8: Preview a sample of review images ────────────────────────────────
import matplotlib.pyplot as plt
import random

review_imgs = sorted(review_dir.glob('*.jpg'))
sample = random.sample(review_imgs, min(6, len(review_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, p in zip(axes.flat, sample, strict=False):
    ax.imshow(Image.open(p))
    ax.set_title(p.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'review_sample.png', dpi=120)
plt.show()
print(f'Sample saved → {OUTPUT_DIR}/review_sample.png')

In [ ]:
# ── Cell 9: Package outputs for download ─────────────────────────────────────
# Zips labels/ and review/ so you can download and open in Label Studio / CVAT
import shutil

archive = OUTPUT_DIR / 'sg_labels_gdino_v1'
shutil.make_archive(str(archive), 'zip', OUTPUT_DIR)
print(f'Archive: {archive}.zip')
print('Download from Drive or use files.download() below.')

# Uncomment to download directly to your browser:
# from google.colab import files
# files.download(str(archive) + '.zip')

## Next Steps

1. Open `review/` in **Label Studio** or **CVAT** — fix misclassifications, add missed vehicles
2. Priority: images flagged as low-confidence in Cell 7 (most ambiguous for the model)
3. Pay special attention to:
   - **Container trucks vs prime movers** — most common confusion near MCE/AYE/Tuas
   - **Motorcycles vs scooters** — hard at distance
   - **Night images** — confidence drops significantly, more review needed
4. Once reviewed, run `notebooks/03_prepare_dataset.ipynb` to build the YOLO training split
5. Then `notebooks/04_train_yolo_baseline.ipynb` with `nc: 10` for the new taxonomy